# Part 4 — Chains of randomness

_Rigorous Courses · Diffusion Models — Part 4 of 12_

**Randomness that unfolds in steps — and why every start ends in the same noise**

You will build a two-state weather chain and check every claim from the lesson against simulation: the transition table, two-step probabilities by enumeration versus matrix multiplication, the stationary distribution, and the Bayes flip that runs the chain backwards. Then you will run the lesson's chain on the number line and watch three very different starting points melt into the same standard Gaussian.

---

This notebook accompanies the lesson. Run cells top to bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

## The weather chain

The lesson's running example: two states (sunny, rainy) and a fixed one-step rule — the **transition kernel**. Each row of the table is a conditional distribution, "given today, here is tomorrow," so each row must sum to 1.

| from \ to | sunny | rainy |
|---|---|---|
| **sunny** | 0.8 | 0.2 |
| **rainy** | 0.4 | 0.6 |

### Step 1 — Write the transition table as an array

Row index = today's state, column index = tomorrow's state, with the coding sunny = 0, rainy = 1. The first thing to check about any kernel: every row sums to 1.

In [ ]:
P = np.array([
    [0.8, 0.2],   # today sunny -> tomorrow [sunny, rainy]
    [0.4, 0.6],   # today rainy -> tomorrow [sunny, rainy]
])
state_names = ["sunny", "rainy"]
row_sums = P.sum(axis=1)

print("transition table P:")
print(P)
print(f"row sums: {row_sums}")

assert np.allclose(row_sums, 1.0), 'each row must be a conditional distribution'

### Step 2 — Simulate the chain and recover the table from data

We simulate 100,000 consecutive days: each day, look up today's row of the table and draw tomorrow from it. Then we go the other way — from the raw day sequence, estimate the transition probabilities by counting: among all days that were sunny, what fraction were followed by rain? If the simulation is faithful, the empirical frequencies should land within about 0.01 of the table entries.

In [ ]:
n_days = 100_000
states = np.zeros(n_days, dtype=int)

for t in range(1, n_days):
    today = states[t - 1]
    p_rainy = P[today, 1]
    states[t] = int(rng.random() < p_rainy)

after_sunny = states[1:][states[:-1] == 0]
after_rainy = states[1:][states[:-1] == 1]
freq_s_to_r = after_sunny.mean()
freq_r_to_r = after_rainy.mean()

print(f"empirical P(rainy tomorrow | sunny today) = {freq_s_to_r:.4f}   (table: {P[0, 1]})")
print(f"empirical P(rainy tomorrow | rainy today) = {freq_r_to_r:.4f}   (table: {P[1, 1]})")

assert abs(freq_s_to_r - P[0, 1]) < 0.01
assert abs(freq_r_to_r - P[1, 1]) < 0.01

### Step 3 — Two-step probabilities: enumerate paths, then let `@` do the bookkeeping

The lesson computed $P(\text{sunny in two days} \mid \text{sunny today}) = 0.8 \cdot 0.8 + 0.2 \cdot 0.4 = 0.72$ by hand: multiply the one-step probabilities along each route, then add the routes. "Multiply along a path, add over paths" is exactly what matrix multiplication computes, so `P @ P` should reproduce every enumerated entry — and its rows should still sum to 1.

In [ ]:
p_ss_2 = P[0, 0] * P[0, 0] + P[0, 1] * P[1, 0]
p_sr_2 = P[0, 0] * P[0, 1] + P[0, 1] * P[1, 1]
P2 = P @ P

print(f"enumeration: P(sunny in 2 | sunny) = {p_ss_2:.4f}")
print(f"enumeration: P(rainy in 2 | sunny) = {p_sr_2:.4f}")
print("matrix product P @ P:")
print(P2)

assert np.isclose(P2[0, 0], p_ss_2)
assert np.isclose(P2[0, 1], p_sr_2)
assert np.allclose(P2.sum(axis=1), 1.0)

## Where chains settle

A distribution $\pi$ is **stationary** when one step of the chain leaves it unchanged: $\pi(s') = \sum_s \pi(s)\, q(s' \mid s)$ for every state $s'$. The lesson solved the weather chain's balance equation and found $\pi = (2/3,\ 1/3)$.

### Step 4 — Check the stationary distribution three ways

First, substitution: feeding $\pi$ through the table (`pi @ P` is the same multiply-and-add bookkeeping as Step 3, applied to a distribution) should return $\pi$ itself. Second, forgetting: push a definitely-sunny start and a definitely-rainy start through 20 steps — both should land on the same distribution. Third, the long run: the fraction of sunny days in our 100,000-day simulation should sit near 2/3.

In [ ]:
pi = np.array([2 / 3, 1 / 3])
pi_next = pi @ P

dist_a = np.array([1.0, 0.0])   # start: definitely sunny
dist_b = np.array([0.0, 1.0])   # start: definitely rainy
for step in range(20):
    dist_a = dist_a @ P
    dist_b = dist_b @ P

frac_sunny = (states == 0).mean()

print(f"pi                 = {pi}")
print(f"pi after one step  = {pi_next}")
print(f"20 steps from sunny start: {dist_a}")
print(f"20 steps from rainy start: {dist_b}")
print(f"long-run fraction of sunny days = {frac_sunny:.4f}   (pi says {2 / 3:.4f})")

assert np.allclose(pi_next, pi)
assert np.allclose(dist_a, dist_b, atol=1e-6)
assert abs(frac_sunny - 2 / 3) < 0.02

## A Markov chain on the number line

Now the chain that matters for diffusion. The state is a real number, and the kernel is a recipe: shrink, then add fresh noise,

$$x_t = \sqrt{1-\beta}\; x_{t-1} + \sqrt{\beta}\; \epsilon_t, \qquad \epsilon_t \sim \mathcal{N}(0, 1),$$

which is the Gaussian kernel $q(x_t \mid x_{t-1}) = \mathcal{N}\big(x_t;\ \sqrt{1-\beta}\,x_{t-1},\ \beta\big)$. The lesson proved the punchline: if $x_{t-1} \sim \mathcal{N}(0,1)$, then $x_t \sim \mathcal{N}(0,1)$ too, because the variance works out to $(1-\beta)\cdot 1 + \beta = 1$. So $\mathcal{N}(0,1)$ is this chain's stationary distribution — the first glimpse of why forward diffusion ends in pure noise.

### Step 5 — Verify the one-step variance algebra for beta = 0.3

Draw 100,000 states from $\mathcal{N}(0,1)$ and 100,000 fresh noises, form the two pieces of the update separately, and measure variances. The algebra predicts: the shrunk state has variance $1-\beta = 0.7$, the scaled noise has variance $\beta = 0.3$, and their sum has variance exactly 1.

In [ ]:
beta = 0.3
n = 100_000
x_prev = rng.standard_normal(n)
eps = rng.standard_normal(n)

shrunk = np.sqrt(1 - beta) * x_prev
noise = np.sqrt(beta) * eps
x_next = shrunk + noise

var_shrunk = shrunk.var()
var_noise = noise.var()
var_next = x_next.var()

print(f"var of sqrt(1-beta) * x_prev = {var_shrunk:.4f}   (algebra: {1 - beta})")
print(f"var of sqrt(beta) * eps      = {var_noise:.4f}   (algebra: {beta})")
print(f"var of the sum               = {var_next:.4f}   (algebra: 1.0)")
print(f"mean of the sum              = {x_next.mean():+.4f}   (algebra: 0.0)")

assert abs(var_shrunk - (1 - beta)) < 0.02
assert abs(var_noise - beta) < 0.02
assert abs(var_next - 1.0) < 0.02
assert abs(x_next.mean()) < 0.02

### Step 6 — Three different starts, one destination

Run 20,000 independent chains from each of three deterministic starts — $x_0 = 4$, $x_0 = -2$, $x_0 = 0$ — for 40 steps. If $\mathcal{N}(0,1)$ really is where the chain settles, all three histograms should sit under the same standard-Gaussian curve, with no memory of where they began.

In [ ]:
n_chains = 20_000
n_steps = 40
starts = [4.0, -2.0, 0.0]
finals = []

for start in starts:
    x = np.full(n_chains, start)
    for step in range(n_steps):
        eps_t = rng.standard_normal(n_chains)
        x = np.sqrt(1 - beta) * x + np.sqrt(beta) * eps_t
    finals.append(x)

grid = np.linspace(-4, 4, 200)
pdf = np.exp(-grid ** 2 / 2) / np.sqrt(2 * np.pi)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, start, x in zip(axes, starts, finals):
    ax.hist(x, bins=60, density=True, color="#4ea1ff", alpha=0.7)
    ax.plot(grid, pdf, color="#ff7b72", linewidth=2, label="N(0,1) pdf")
    ax.set_title(f"start x0 = {start:g}, after {n_steps} steps")
    ax.set_xlabel("x")
axes[0].set_ylabel("density")
axes[0].legend()
plt.tight_layout()
plt.show()

for start, x in zip(starts, finals):
    print(f"start {start:+5.1f}: final mean = {x.mean():+.4f}, final var = {x.var():.4f}")

for x in finals:
    assert abs(x.mean()) < 0.05
    assert abs(x.var() - 1.0) < 0.05

### Step 7 — Watch the memory of the start decay

The lesson showed the mean obeys $m_t = \sqrt{1-\beta}\, m_{t-1}$, so from $x_0 = 4$ it should trace the geometric curve $4\,(1-\beta)^{t/2}$. We track the empirical mean of 20,000 chains at every step and overlay the prediction — the two curves should be indistinguishable.

In [ ]:
x = np.full(n_chains, 4.0)
mean_track = [x.mean()]

for step in range(n_steps):
    eps_t = rng.standard_normal(n_chains)
    x = np.sqrt(1 - beta) * x + np.sqrt(beta) * eps_t
    mean_track.append(x.mean())

steps_axis = np.arange(n_steps + 1)
predicted_mean = 4.0 * (1 - beta) ** (steps_axis / 2)
max_gap = np.max(np.abs(np.array(mean_track) - predicted_mean))

plt.figure(figsize=(7, 4))
plt.plot(steps_axis, mean_track, "o", markersize=4, color="#4ea1ff", label="simulated mean")
plt.plot(steps_axis, predicted_mean, color="#ff7b72", linewidth=2, label="predicted 4*(1-beta)^(t/2)")
plt.xlabel("step t")
plt.ylabel("mean of x_t")
plt.title("the chain forgets its start geometrically")
plt.legend()
plt.show()

print(f"largest gap between simulated and predicted mean = {max_gap:.4f}")

assert max_gap < 0.05

## Running the chain backwards

Generation needs "yesterday given today," and Bayes' rule delivers it:

$$p(x_{t-1} \mid x_t) = \frac{q(x_t \mid x_{t-1})\; p(x_{t-1})}{p(x_t)}.$$

The lesson's worked example: prior $p(\text{yesterday} = \text{sunny}) = 0.9$, today it rains. Hand computation: joint sunny-then-rain $0.9 \times 0.2 = 0.18$; joint rainy-then-rain $0.1 \times 0.6 = 0.06$; marginal of rain $0.24$; flip $0.18 / 0.24 = 0.75$.

### Step 8 — The Bayes flip, empirically

Simulate 200,000 (yesterday, today) pairs: draw yesterday from the 0.9/0.1 prior, then step the weather chain once. Keep only the pairs where today is rainy, and ask: in what fraction was yesterday sunny? That conditional frequency is the empirical Bayes flip, and it should match the hand answer 0.75. Notice the flip used the **prior**, not only the table — change the prior and the answer changes (practice problem 5).

In [ ]:
n_pairs = 200_000
p_sunny_yesterday = 0.9

yesterday_sunny = rng.random(n_pairs) < p_sunny_yesterday
p_rain_today = np.where(yesterday_sunny, P[0, 1], P[1, 1])
today_rainy = rng.random(n_pairs) < p_rain_today

empirical_flip = yesterday_sunny[today_rainy].mean()

hand_numerator = P[0, 1] * p_sunny_yesterday
hand_marginal = P[0, 1] * p_sunny_yesterday + P[1, 1] * (1 - p_sunny_yesterday)
hand_flip = hand_numerator / hand_marginal

print(f"hand Bayes: p(yesterday sunny | today rainy) = {hand_flip:.4f}")
print(f"simulation: p(yesterday sunny | today rainy) = {empirical_flip:.4f}")
print(f"rainy days used for the estimate: {today_rainy.sum()}")

assert abs(empirical_flip - hand_flip) < 0.01

## Practice

The same problems as the lesson. Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Factorize the joint $p(a, b, c)$ two ways: forward (peeling $c$ last) and backward (peeling $a$ last). Then write what each factorization becomes if the process is Markov with step order $a \to b \to c$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Forward, peel off $c$: by the definition of the conditional, $p(a,b,c) = p(a,b)\,p(c \mid a,b)$.
- Peel off $b$ the same way: $p(a,b) = p(a)\,p(b \mid a)$.
- Substitute (equals may replace equals): $p(a,b,c) = p(a)\,p(b \mid a)\,p(c \mid a,b)$.
- Backward, peel off $a$: $p(a,b,c) = p(b,c)\,p(a \mid b,c)$.
- Peel off $b$ from the remaining pair and substitute: $p(b,c) = p(c)\,p(b \mid c)$, so $p(a,b,c) = p(c)\,p(b \mid c)\,p(a \mid b,c)$. The chain rule is bookkeeping — the peeling order is free.
- Markov, forward telling: $p(c \mid a,b) = p(c \mid b)$, so $p(a)\,p(b \mid a)\,p(c \mid b)$.
- Markov, backward telling: given the present $b$, past and future are independent, so $p(a \mid b,c) = p(a \mid b)$, giving $p(c)\,p(b \mid c)\,p(a \mid b)$.

**Answer:** forward $p(a)\,p(b \mid a)\,p(c \mid a,b) \to p(a)\,p(b \mid a)\,p(c \mid b)$; backward $p(c)\,p(b \mid c)\,p(a \mid b,c) \to p(c)\,p(b \mid c)\,p(a \mid b)$ — every factor one-step.

</details>

**Problem 2.** A three-day weather record (letters are day 1, day 2, day 3; S = sunny, R = rainy) has path probabilities: SSS 0.36, SSR 0.04, SRS 0.04, SRR 0.06, RSS 0.10, RSR 0.10, RRS 0.12, RRR 0.18. Is this process Markov? Decide by computing $p(\text{day 3} = \text{S} \mid \text{day 2} = \text{S}, \text{day 1} = \text{S})$ and $p(\text{day 3} = \text{S} \mid \text{day 2} = \text{S}, \text{day 1} = \text{R})$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

Condition on the history, one slice at a time:

- History SS: matching paths SSS (0.36) and SSR (0.04), slice mass 0.40. Conditional: $0.36 / 0.40 = 0.90$.
- History RS: matching paths RSS (0.10) and RSR (0.10), slice mass 0.20. Conditional: $0.10 / 0.20 = 0.50$.
- $0.90 \ne 0.50$: day 1 still changes the day-3 forecast after day 2 is known, so the process is **not Markov**.

```python
paths = {"SSS": 0.36, "SSR": 0.04, "SRS": 0.04, "SRR": 0.06,
         "RSS": 0.10, "RSR": 0.10, "RRS": 0.12, "RRR": 0.18}
p3_given_SS = paths["SSS"] / (paths["SSS"] + paths["SSR"])
p3_given_RS = paths["RSS"] / (paths["RSS"] + paths["RSR"])
print(p3_given_SS, p3_given_RS)   # 0.9  0.5
```

**Answer:** not Markov — 0.90 versus 0.50; a single mismatch breaks the Markov property.

</details>

**Problem 3.** A chain has kernel $P(\text{S} \to \text{S}) = 0.7$, $P(\text{S} \to \text{R}) = 0.3$, $P(\text{R} \to \text{S}) = 0.5$, $P(\text{R} \to \text{R}) = 0.5$. Starting sunny, compute the probability of sunny two steps later.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Route through sunny: $0.7 \times 0.7 = 0.49$ (multiply one-step probabilities along the path).
- Route through rainy: $0.3 \times 0.5 = 0.15$.
- Add the disjoint routes: $0.49 + 0.15 = 0.64$.
- Check the complement: $0.7 \times 0.3 + 0.3 \times 0.5 = 0.36$, and $0.64 + 0.36 = 1$.

```python
Q = np.array([[0.7, 0.3], [0.5, 0.5]])
Q2 = Q @ Q
print(Q2[0, 0])   # 0.64
```

**Answer:** $0.64$.

</details>

**Problem 4.** Let $\beta = 0.08$. Show that if $x_{t-1} \sim \mathcal{N}(0,1)$ and $\epsilon_t \sim \mathcal{N}(0,1)$ is independent of it, then $x_t = \sqrt{0.92}\, x_{t-1} + \sqrt{0.08}\, \epsilon_t$ is again $\mathcal{N}(0,1)$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Scaling rule (part 3): $\sqrt{0.92}\,x_{t-1} \sim \mathcal{N}(0,\ 0.92)$ — the mean scales by $a$, the variance by $a^2 = 0.92$.
- Scaling rule again: $\sqrt{0.08}\,\epsilon_t \sim \mathcal{N}(0,\ 0.08)$.
- Sum rule for independent Gaussians: means add ($0 + 0 = 0$), variances add ($0.92 + 0.08 = 1$), and the result is exactly Gaussian.

```python
b = 0.08
xa = rng.standard_normal(200_000)
ea = rng.standard_normal(200_000)
ya = np.sqrt(1 - b) * xa + np.sqrt(b) * ea
print(ya.mean(), ya.var())   # close to 0 and 1
```

**Answer:** $x_t \sim \mathcal{N}(0,1)$ exactly — the shrink and the dose are calibrated so $(1-\beta) + \beta = 1$ for any $\beta$.

</details>

**Problem 5.** Same weather kernel as the lesson ($\text{S} \to \text{R}$ has probability 0.2, $\text{R} \to \text{R}$ has probability 0.6), but now the prior is even odds: $p(\text{yesterday} = \text{S}) = 0.5$. Today it rains. Compute $p(\text{yesterday} = \text{S} \mid \text{today} = \text{R})$ and compare with the lesson's answer under the 0.9 prior.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Joint sunny-then-rain: $0.5 \times 0.2 = 0.10$.
- Joint rainy-then-rain: $0.5 \times 0.6 = 0.30$.
- Marginal of rain: $0.10 + 0.30 = 0.40$.
- Bayes flip: $0.10 / 0.40 = 0.25$.

```python
numer = 0.5 * 0.2
denom = 0.5 * 0.2 + 0.5 * 0.6
print(numer / denom)   # 0.25
```

**Answer:** $0.25$, versus $0.75$ under the 0.9 prior. Same forward table, different prior, different reverse conditional — the flip needs the marginal of the earlier state, which is exactly why diffusion's reverse step cannot be read off the noising kernel alone.

</details>

**Problem 6.** Why does the noising chain forget its start? Make it quantitative: with $\beta = 0.3$ and the deterministic start $x_0 = 10$, find the mean and variance of $x_{20}$, and state the general principle.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Mean recursion: $m_t = \sqrt{0.7}\,m_{t-1}$ (expectation is linear and the noise has mean 0), so $m_{20} = 0.7^{10} \times 10 \approx 0.28$.
- Variance recursion: $v_t = 0.7\,v_{t-1} + 0.3$ from $v_0 = 0$; its closed form $v_t = 1 - 0.7^{t}$ satisfies both the recursion and the start, so $v_{20} = 1 - 0.7^{20} \approx 0.9992$.
- So $x_{20} \approx \mathcal{N}(0.28,\ 0.999)$ — essentially standard noise, despite starting at 10.

```python
m20 = 10 * 0.7 ** 10
v20 = 1 - 0.7 ** 20
print(m20, v20)   # 0.282...  0.9992...
```

**Answer:** mean $\approx 0.28$, variance $\approx 0.999$. The start's influence decays like $(1-\beta)^{t/2}$ — geometrically fast — while the injected noise rebuilds the variance toward 1, so every start flows to the same $\mathcal{N}(0,1)$.

</details>

## Wrap-up

Verified in this notebook: the transition table is recoverable from simulated data; two-step probabilities from path enumeration equal the matrix product `P @ P`; the weather chain's stationary distribution $(2/3,\ 1/3)$ reproduces itself, attracts both extreme starts, and matches the long-run sunny fraction; the number-line chain's one-step variance algebra $(1-\beta) + \beta = 1$ holds numerically; three different starts all settle into $\mathcal{N}(0,1)$ with the mean decaying along the predicted $(1-\beta)^{t/2}$ curve; and the empirical Bayes flip $p(\text{yesterday} \mid \text{today})$ matches the hand computation 0.75. Part 5 builds the scoring toolkit — likelihood, KL divergence, and the ELBO — the single number that will tell us how good a learned reverse chain is.